# Estructura del notebook
1. Realizar benchmark de generacion de trazas no procesadas (timestamp, hex data separadas por espacios)
2. Separacion de datos por codigo de funcion

## Generador de benchmark

In [1]:
import pandas as pd
import modbus_parser
import time
import glob
from datetime import datetime

In [ ]:

# Buscar y unir todos los archivos crudos en el directorio de entrada
input_files = glob.glob("./data/input/raw_*.csv")
print(f"Cargando datos en memoria desde {len(input_files)} archivos...\n")

# Concatenar todos los archivos encontrados en un solo DataFrame
df_raw = pd.concat((pd.read_csv(f) for f in input_files), ignore_index=True)
df_raw = df_raw.dropna(subset=['hex_data', 'timestamp'])

fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")

In [ ]:
# IMPORTANTE LEER
# Si es verdadero, genera el archivo procesado final en output_file
PROCESS_DATA = True
output_file = f"./data/output/processed_traces/pr_ts-{fecha_actual}.csv"

In [ ]:
# ---------------------------------------------------------
# Prueba 1: Secuencial (Bucle For en Python)
# ---------------------------------------------------------
print("Ejecutando método 1 (Secuencial)...")
start_seq = time.perf_counter()

resultados_validos = []
for index, row in df_raw.iterrows():
    timestamp = row['timestamp']
    hex_string = row['hex_data']

    try:
        raw_bytes = bytes.fromhex(hex_string.replace(" ", ""))
        resultado = modbus_parser.process_single_trace(raw_bytes)
        if resultado is not None:
            resultado['timestamp'] = timestamp
            resultados_validos.append(resultado)
    except Exception:
        pass

if resultados_validos:
    df_seq = pd.DataFrame(resultados_validos)
    df_seq['timestamp'] = pd.to_datetime(df_seq['timestamp'])
    df_seq['delta_t'] = df_seq['timestamp'].diff().dt.total_seconds().fillna(0.0)

end_seq = time.perf_counter()
time_seq = end_seq - start_seq

# ---------------------------------------------------------
# Prueba 2: Paralelo en Batch (Rust + Rayon)
# ---------------------------------------------------------
print("Ejecutando método 2 (Paralelo / Batch)...")
start_par = time.perf_counter()

timestamps_list = df_raw['timestamp'].astype(str).tolist()
hex_list = df_raw['hex_data'].tolist()

resultado_dict = modbus_parser.process_batch_traces(timestamps_list, hex_list)
df_par = pd.DataFrame(resultado_dict)

if not df_par.empty:
    df_par['timestamp'] = pd.to_datetime(df_par['timestamp'])
    df_par['delta_t'] = df_par['timestamp'].diff().dt.total_seconds().fillna(0.0)
    
    # Exportar resultados paralelos si la bandera está activada
    if PROCESS_DATA:
        df_par.to_csv(output_file, index=False)
        print(f"\n[+] Resultados exportados exitosamente a: {output_file}")

end_par = time.perf_counter()
time_par = end_par - start_par

print(f"\nTiempo Secuencial : {time_seq:.4f} segundos")
print(f"Tiempo Paralelo   : {time_par:.4f} segundos")

if time_par > 0:
    print(f"Relacion tiempo secuencial / tiempo paralelo: {time_seq / time_par:.2f}")

## Separación

In [2]:
import pandas as pd
import glob
import os
from datetime import datetime

# Buscar todos los archivos de trazas procesadas
ruta_busqueda = "./data/output/processed_traces/pr_ts-*.csv"
archivos = sorted(glob.glob(ruta_busqueda), key=os.path.getmtime, reverse=True)

if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos en {ruta_busqueda}")

# Seleccionar el archivo más reciente (el primero tras ordenar inversamente por tiempo)
archivo_reciente = archivos[0]
print(f"[+] Cargando el archivo más reciente: {archivo_reciente}")

# Cargar los datos en memoria
df_principal = pd.read_csv(archivo_reciente)
print(f"[+] Total de trazas cargadas: {len(df_principal)}")

[+] Cargando el archivo más reciente: ./data/output/processed_traces\pr_ts-20260913_014748.csv
[+] Total de trazas cargadas: 18525


In [3]:
import numpy as np

# 1. Crear el directorio de separación con la fecha actual
fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
directorio_salida = f"./data/separations/{fecha_actual}"
os.makedirs(directorio_salida, exist_ok=True)
print(f"[+] Directorio creado: {directorio_salida}\n")

# Obtener todos los códigos de función únicos en el dataset
codigos_presentes = df_principal['function_code'].dropna().unique()

for fc in codigos_presentes:
    # 2. Aislar las trazas de este código de función específico
    df_fc = df_principal[df_principal['function_code'] == fc].copy()
    
    # 3. Eliminar columnas que contengan exclusivamente valores -1
    # Se evalúa columna por columna: si TODOS sus datos son -1, se descarta para este FC
    columnas_validas = [col for col in df_fc.columns if not (df_fc[col] == -1).all()]
    df_fc_limpio = df_fc[columnas_validas]
    
    # 4. Generar y guardar el archivo CSV
    archivo_out = f"{directorio_salida}/dataset_fc_{int(fc)}.csv"
    df_fc_limpio.to_csv(archivo_out, index=False)
    
    # Imprimir el resumen de la limpieza
    columnas_eliminadas = len(df_fc.columns) - len(df_fc_limpio.columns)
    print(f" -> FC {int(fc):02d}: {len(df_fc_limpio)} trazas retenidas.")
    print(f"          Columnas activas: {len(df_fc_limpio.columns)} (Se eliminaron {columnas_eliminadas} columnas vacías)")

print(f"\n[+] Separación completada. Todos los archivos están en: {directorio_salida}")

[+] Directorio creado: ./data/separations/20260913_133731

 -> FC 04: 2958 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 16: 1482 trazas retenidas.
          Columnas activas: 30 (Se eliminaron 2 columnas vacías)
 -> FC 03: 5899 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 01: 2942 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 02: 2954 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 05: 872 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 06: 1418 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)

[+] Separación completada. Todos los archivos están en: ./data/separations/20260913_133731


## Entrenamiento por codigo de funcion
Pasos: 
Separacion de codigo de funcion
Implementacio de correlacion por codigo de funcion
Implementacion de Isolation Forest

### Separacion y analisis de correlacion por codigo de funcion 1-6

In [15]:
import pandas as pd
import glob
import os
import numpy as np

def analizar_autocorrelacion_lectura(fc_objetivo, ruta_busqueda="./data/output/processed_traces/pr_ts-*.csv"):
    """
    Escanea el archivo procesado más reciente y calcula la autocorrelación 
    específicamente para los códigos de función del 1 al 6.
    """
    if not (1 <= fc_objetivo <= 6):
        print(f"[-] Advertencia: El código {fc_objetivo} está fuera del rango 1-6 para esta función.")
        
    # 1. Buscar y cargar el archivo procesado más reciente
    archivos = sorted(glob.glob(ruta_busqueda), key=os.path.getmtime, reverse=True)

    if not archivos:
        raise FileNotFoundError("No se encontraron archivos procesados.")

    df_principal = pd.read_csv(archivos[0])

    # 2. Filtrar únicamente el Código de Función especificado
    df_fc = df_principal[df_principal['function_code'] == fc_objetivo].copy()

    if df_fc.empty:
        print(f"[-] No hay trazas con Function Code {fc_objetivo} en el archivo.")
        return
        
    print(f"[+] Analizando {len(df_fc)} trazas del Function Code {fc_objetivo}...")
    
    # Asegurar el orden cronológico
    df_fc['timestamp'] = pd.to_datetime(df_fc['timestamp'])
    df_fc = df_fc.sort_values('timestamp')
    
    # Columnas exclusivas para análisis de lectura/escritura simple
    columnas_analisis = ['address_unit_1', 'quantity_unit_1', 'delta_t']
    
    def encontrar_mejor_patron(serie, max_lag=30):
        if serie.std() == 0 or len(serie.dropna()) < max_lag:
            return None, 0.0
            
        mejor_lag = 0
        mejor_score = -1.0
        
        for lag in range(1, max_lag + 1):
            score = serie.autocorr(lag=lag)
            if not np.isnan(score) and score > mejor_score:
                mejor_score = score
                mejor_lag = lag
                
        return mejor_lag, mejor_score

    print(f"\n=== Resultados de Autocorrelación para FC {fc_objetivo} ===")
    
    for col in columnas_analisis:
        if col in df_fc.columns:
            # Reemplazar posibles -1 con NaN para que no distorsionen la matemática
            serie_limpia = df_fc[col].replace(-1, np.nan).dropna()
            
            n_estimado, score = encontrar_mejor_patron(serie_limpia)
            
            print(f"Variable: {col}")
            if n_estimado:
                print(f"  └─ Ventana N detectada : {n_estimado}")
                print(f"  └─ Score de confianza  : {score:.4f} (-1 a 1)")
                
                # Evaluación de utilidad del patrón
                if score > 0.8:
                    print("  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)")
                elif score > 0.4:
                    print("  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)")
                else:
                    print("  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)")
            else:
                print("  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).")
            print("-" * 40)

In [16]:
analizar_autocorrelacion_lectura(1)

[+] Analizando 2942 trazas del Function Code 1...

=== Resultados de Autocorrelación para FC 1 ===
Variable: address_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9973 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9973 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9431 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [17]:
analizar_autocorrelacion_lectura(2)

[+] Analizando 2954 trazas del Function Code 2...

=== Resultados de Autocorrelación para FC 2 ===
Variable: address_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9959 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9959 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.8797 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [18]:
analizar_autocorrelacion_lectura(3)

[+] Analizando 5899 trazas del Function Code 3...

=== Resultados de Autocorrelación para FC 3 ===
Variable: address_unit_1
  └─ Ventana N detectada : 4
  └─ Score de confianza  : 0.9923 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 4
  └─ Score de confianza  : 0.9230 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 4
  └─ Score de confianza  : 0.9506 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [19]:
analizar_autocorrelacion_lectura(4)

[+] Analizando 2958 trazas del Function Code 4...

=== Resultados de Autocorrelación para FC 4 ===
Variable: address_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9973 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 16
  └─ Score de confianza  : 0.6409 (-1 a 1)
  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9565 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [20]:
analizar_autocorrelacion_lectura(5)

[+] Analizando 872 trazas del Function Code 5...

=== Resultados de Autocorrelación para FC 5 ===
Variable: address_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 1
  └─ Score de confianza  : 0.5012 (-1 a 1)
  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9174 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [21]:
analizar_autocorrelacion_lectura(6)

[+] Analizando 1418 trazas del Function Code 6...

=== Resultados de Autocorrelación para FC 6 ===
Variable: address_unit_1
  └─ Ventana N detectada : 1
  └─ Score de confianza  : 0.3282 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 1
  └─ Score de confianza  : 0.4515 (-1 a 1)
  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 18
  └─ Score de confianza  : 0.5965 (-1 a 1)
  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)
----------------------------------------


### Separacion y analisis de correlacion por codigo de funcion 15, 16 y 23

In [5]:
import pandas as pd
import glob
import os
import numpy as np

# 1. Buscar y cargar el archivo procesado más reciente
ruta_busqueda = "./data/output/processed_traces/pr_ts-*.csv"
archivos = sorted(glob.glob(ruta_busqueda), key=os.path.getmtime, reverse=True)

if not archivos:
    raise FileNotFoundError("No se encontraron archivos procesados.")

df_principal = pd.read_csv(archivos[0])

# 2. Filtrar únicamente el Código de Función 16
df_fc16 = df_principal[df_principal['function_code'] == 16].copy()

if df_fc16.empty:
    print("[-] No hay trazas con Function Code 16 en el archivo.")
else:
    print(f"[+] Analizando {len(df_fc16)} trazas del Function Code 16...")
    
    # Asegurar el orden cronológico
    df_fc16['timestamp'] = pd.to_datetime(df_fc16['timestamp'])
    df_fc16 = df_fc16.sort_values('timestamp')
    
    # Seleccionar todas las columnas excepto las metadatas especificadas
    cols_excluir = ['timestamp', 'slave_address', 'function_code', 'function_name', 'crc_calculated']
    columnas_analisis = [col for col in df_fc16.columns if col not in cols_excluir]
    
    def encontrar_mejor_patron(serie, max_lag=30):
        """Calcula la autocorrelación hasta max_lag y devuelve el pico más alto."""
        if serie.std() == 0 or len(serie.dropna()) < max_lag:
            return None, 0.0
            
        mejor_lag = 0
        mejor_score = -1.0
        
        for lag in range(1, max_lag + 1):
            score = serie.autocorr(lag=lag)
            if not np.isnan(score) and score > mejor_score:
                mejor_score = score
                mejor_lag = lag
                
        return mejor_lag, mejor_score

    print("\n=== Resultados de Autocorrelación para FC 16 ===")
    
    for col in columnas_analisis:
        # Reemplazar posibles -1 con NaN
        serie_limpia = df_fc16[col].replace(-1, np.nan).dropna()
        
        n_estimado, score = encontrar_mejor_patron(serie_limpia)
        
        print(f"Variable: {col}")
        if n_estimado:
            print(f"  └─ Ventana N detectada : {n_estimado}")
            print(f"  └─ Score de confianza  : {score:.4f} (-1 a 1)")
            
            # Evaluación de utilidad del patrón
            if score > 0.8:
                print("  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)")
            elif score > 0.4:
                print("  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)")
            else:
                print("  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)")
        else:
            print("  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).")
        print("-" * 40)

[+] Analizando 1482 trazas del Function Code 16...

=== Resultados de Autocorrelación para FC 16 ===
Variable: address_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: quantity_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: count_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: mininum_value_register
  └─ Ventana N detectada : 29
  └─ Score de confianza  : 0.0596 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: maximum_value_register
  └─ Ventana N detectada : 9
  └─ Score de confianza  : 0.0483 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: tmean_value_register
  └─ Ventana N d

## Funciones de correlacion

Para codigo de funcion 15, 16 y 23

In [11]:
import pandas as pd
import glob
import os
import numpy as np

def analizar_autocorrelacion_fc(fc_objetivo, ruta_busqueda="./data/output/processed_traces/pr_ts-*.csv"):
    """
    Escanea el archivo procesado más reciente, aísla un código de función
    y calcula el mejor 'lag' de autocorrelación para todas sus variables útiles.
    """
    # 1. Buscar y cargar el archivo procesado más reciente
    archivos = sorted(glob.glob(ruta_busqueda), key=os.path.getmtime, reverse=True)

    if not archivos:
        raise FileNotFoundError("No se encontraron archivos procesados.")

    df_principal = pd.read_csv(archivos[0])

    # 2. Filtrar únicamente el Código de Función especificado
    df_fc = df_principal[df_principal['function_code'] == fc_objetivo].copy()

    if df_fc.empty:
        print(f"[-] No hay trazas con Function Code {fc_objetivo} en el archivo.")
        return
        
    print(f"[+] Analizando {len(df_fc)} trazas del Function Code {fc_objetivo}...")
    
    # Asegurar el orden cronológico
    df_fc['timestamp'] = pd.to_datetime(df_fc['timestamp'])
    df_fc = df_fc.sort_values('timestamp')
    
    # Seleccionar todas las columnas excepto las metadatas especificadas
    cols_excluir = ['timestamp', 'slave_address', 'function_code', 'function_name', 'crc_calculated']
    columnas_analisis = [col for col in df_fc.columns if col not in cols_excluir]
    
    def encontrar_mejor_patron(serie, max_lag=30):
        if serie.std() == 0 or len(serie.dropna()) < max_lag:
            return None, 0.0
            
        mejor_lag = 0
        mejor_score = -1.0
        
        for lag in range(1, max_lag + 1):
            score = serie.autocorr(lag=lag)
            if not np.isnan(score) and score > mejor_score:
                mejor_score = score
                mejor_lag = lag
                
        return mejor_lag, mejor_score

    print(f"\n=== Resultados de Autocorrelación para FC {fc_objetivo} ===")
    
    for col in columnas_analisis:
        # Reemplazar posibles -1 con NaN
        serie_limpia = df_fc[col].replace(-1, np.nan).dropna()
        
        n_estimado, score = encontrar_mejor_patron(serie_limpia)
        
        print(f"Variable: {col}")
        if n_estimado:
            print(f"  └─ Ventana N detectada : {n_estimado}")
            print(f"  └─ Score de confianza  : {score:.4f} (-1 a 1)")
            
            # Evaluación de utilidad del patrón
            if score > 0.8:
                print("  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)")
            elif score > 0.4:
                print("  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)")
            else:
                print("  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)")
        else:
            print("  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).")
        print("-" * 40)

In [12]:
analizar_autocorrelacion_fc(15)

[-] No hay trazas con Function Code 15 en el archivo.


In [13]:
analizar_autocorrelacion_fc(16)

[+] Analizando 1482 trazas del Function Code 16...

=== Resultados de Autocorrelación para FC 16 ===
Variable: address_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: quantity_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: count_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: mininum_value_register
  └─ Ventana N detectada : 29
  └─ Score de confianza  : 0.0596 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: maximum_value_register
  └─ Ventana N detectada : 9
  └─ Score de confianza  : 0.0483 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: tmean_value_register
  └─ Ventana N d

In [14]:
analizar_autocorrelacion_fc(23)

[-] No hay trazas con Function Code 23 en el archivo.
